## Compute seasonal NDVI composite from cloud-masked, water-masked, Sentinel-2 SR

In [1]:
import ee
import geemap

from pprint import pprint as pp

In [2]:
# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project= "charrell-personal")

#Initialize Map
Map = geemap.Map()
Map.add_layer_manager()

#### Define user variables

In [3]:
AOI = ee.Geometry.Point([37.42638181390271,0.23409463839546163])

LLBN = ee.Geometry.Polygon([
          [
            [
              37.3290089891249,
              0.3099181340367778
            ],
            [
              37.3290089891249,
              0.11712184774413004
            ],
            [
              37.54649878002513,
              0.11712184774413004
            ],
            [
              37.54649878002513,
              0.3099181340367778
            ],
            [
              37.3290089891249,
              0.3099181340367778
            ]
          ]
        ])

START_DATE = ee.Date("2025-03-01")
END_DATE = START_DATE.advance(90, "day")

# CLOUD MASKING PARAMETERS
CLOUD_FILTER = 70     # % max CLOUDY_PIXEL_PERCENTAGE per image
CLD_PRB_THRESH = 40     # % s2cloudless probability threshold
NIR_DRK_THRESH = 0.15   # reflectance (0..1) threshold for dark pixels
CLD_PRJ_DIST_KM = 1.0    # max shadow search distance (km)
BUFFER_M = 50     # dilation buffer for cloud+shadow mask (m)
ERODE_RADIUS_M = 40     # small erosion to denoise speckle (m)

# Performance for heavy ops
DDT_SCALE_M = 100    # scale used for directionalDistanceTransform (m)
MORPH_SCALE_M = 20     # scale used for focal ops (m)

# WATER MASKING PARAMETERS
SEASONALITY_MONTHS = 10 # Threshold (0–12) for permanent water from JRC.
DW_PROB_THRESHOLD = 0.5 # Probability threshold (0–1) for Dynamic World's 'water' band.


#### Define helper functions

In [4]:
def mask_edges(image):
    """Masks edge pixels for 20m and 60m bands"""
    return (image
            .updateMask(image.select('B8A').mask())
            .updateMask(image.select('B9').mask()))

def add_ndvi(image):
    """Calculates the Normalized Difference Vegetation Index (NDVI) from NIR and Red bands."""
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)

def _validate_s2_sr_date_range(aoi, start_date, end_date):
    """
    Raises ValueError if the provided date window does not overlap the available
    COPERNICUS/S2_SR_HARMONIZED imagery over the AOI, or if start_date >= end_date.

    Assumes start_date and end_date are ee.Date.
    """
    col = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(aoi)

    # No imagery over AOI
    if col.size().getInfo() == 0:
        raise ValueError("No Sentinel-2 SR imagery found for the provided AOI.")

    # Dataset temporal bounds (ms since epoch) over AOI
    min_ts = col.aggregate_min('system:time_start').getInfo()
    max_ts = col.aggregate_max('system:time_start').getInfo()
    min_str = ee.Date(min_ts).format('YYYY-MM-dd').getInfo()
    max_str = ee.Date(max_ts).format('YYYY-MM-dd').getInfo()

    # Convert user inputs (ms + strings)
    s_ms  = start_date.millis().getInfo()
    e_ms  = end_date.millis().getInfo()
    s_str = start_date.format('YYYY-MM-dd').getInfo()
    e_str = end_date.format('YYYY-MM-dd').getInfo()

    if s_ms >= e_ms:
        raise ValueError(f"Invalid date range: start_date ({s_str}) must be earlier than end_date ({e_str}).")

    # Must overlap dataset window: [start, end) vs [min_ts, max_ts]
    # Start must be < max_ts (i.e. before the most recent image)
    if s_ms >= max_ts:
        raise ValueError(f"Invalid start_date: {s_str} is on/after the most recent S2_SR date over this AOI ({max_str}).")
    # End must be > min_ts (i.e. after the earliest image)
    if e_ms <= min_ts:
        raise ValueError(f"Invalid end_date: {e_str} is on/before the earliest S2_SR date over this AOI ({min_str}).")
    # Keep bounds within dataset window for stricter validity
    if s_ms < min_ts:
        raise ValueError(f"Invalid start_date: {s_str} is earlier than the first available S2_SR date over this AOI ({min_str}).")
    if e_ms > max_ts:
        raise ValueError(f"Invalid end_date: {e_str} is later than the most recent S2_SR date over this AOI ({max_str}).")


#### Import Datasets

Harmonized Sentinel-2 MSI: MultiSpectral Instrument, Level-2A
```ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")```

Sentinel-2: Cloud Probability
```ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")```

JRC Global Surface Water Mapping Layers
```ee.Image('JRC/GSW1_4/GlobalSurfaceWater')```

Dynamic World V1
```ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")```

The cloud masking function utilizes [COPERNICUS/S2_CLOUD_PROBABILITY](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_CLOUD_PROBABILITY).
See [this tutorial](https://developers.google.com/earth-engine/tutorials/community/sentinel-2-s2cloudless) explaining how to apply the cloud mask. This image
collection contains a single 1-100 'probability' band representing the probability that a pixel is cloudy.

#### Filter and join Sentinel-2 and Sentinel-2 Cloud Probability Image Collections

In [5]:
def get_s2_sr_cld_col(aoi, start_date, end_date):
    """
    Join S2_SR_HARMONIZED with S2_CLOUD_PROBABILITY (by system:index), filtered to AOI/dates.

    Args:
        aoi (ee.Geometry)
        start_date (ee.Date)
        end_date (ee.Date)

    Returns:
        ee.ImageCollection: A filtered collection of S2_SR_HAR and S2_CLOUD_PROB collections
    """
    # Validate dates
    _validate_s2_sr_date_range(aoi, start_date, end_date)

    # Import and filter S2 SR.
    s2_sr_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
        .map(mask_edges))

    # Import and filter s2cloudless.
    s2_cloudless_col = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
        .filterBounds(aoi)
        .filterDate(start_date, end_date))

    # Join the filtered s2cloudless collection to the SR collection by the 'system:index' property.
    joined = ee.ImageCollection(ee.Join.saveFirst('s2cloudless').apply(**{
        'primary': s2_sr_col,
        'secondary': s2_cloudless_col,
        'condition': ee.Filter.equals(leftField='system:index', rightField='system:index')
    }))

    return joined


#### Define Cloud Masking Functions

In [6]:

def add_cld_shdw_mask(img):
    """
    Build a cloud+shadow mask for a Sentinel-2 SR image.

    Uses s2cloudless cloud probability (thresholded), dark NIR pixels (reflectance)
    excluding water (SCL == 6), and a directional shadow projection along the solar
    azimuth. Applies an opening (erosion then dilation) in meters to clean/grow the mask.

    Args:
        img (ee.Image): A COPERNICUS/S2_SR_HARMONIZED image joined with an s2cloudless
            image (accessible via img.get('s2cloudless')) and including the SCL band.

    Returns:
        img (ee.Image): The original image with an added 'cloudmask' band (1 = cloud|shadow, 0 = clear)."""
    # Cloud mask from s2cloudless
    cld_prb = ee.Image(img.get('s2cloudless')).select('probability')
    is_cloud = cld_prb.gt(CLD_PRB_THRESH)

    # Dark pixels (NIR) excluding water (SCL == 6)
    not_water = img.select('SCL').neq(6)
    nir_refl = img.select('B8').multiply(0.0001)
    dark_pixels = nir_refl.lt(NIR_DRK_THRESH).And(not_water)

    # Project shadows from clouds along solar azimuth
    # Angle in DEGREES, distance in PIXELS (per docs). Reproject the cloud mask
    # to a coarse grid so "pixels" == DDT_SCALE_M, then convert km -> pixels.
    shadow_azimuth_deg = ee.Number(90).subtract(ee.Number(img.get('MEAN_SOLAR_AZIMUTH_ANGLE')))
    max_dist_px = ee.Number(CLD_PRJ_DIST_KM).multiply(1000.0).divide(DDT_SCALE_M).int()

    clouds_for_ddt = is_cloud.reproject(crs=img.select(0).projection(), scale=DDT_SCALE_M)
    cld_proj = (clouds_for_ddt.directionalDistanceTransform(shadow_azimuth_deg, max_dist_px)
                .select('distance')
                .mask())

    is_shadow = cld_proj.And(dark_pixels)

    # Combine cloud + shadow then apply opening (erode → dilate) in METERS
    cld_or_shdw = is_cloud.Or(is_shadow)
    opened = (cld_or_shdw
              .reproject(crs=img.select(0).projection(), scale=MORPH_SCALE_M)
              .focalMin(ERODE_RADIUS_M, units='meters')
              .focalMax(BUFFER_M, units='meters'))

    return img.addBands(opened.rename('cloudmask'))


def apply_cld_shdw_mask(img):
    """Invert 'cloudmask' and apply it to ALL bands."""
    clear = img.select('cloudmask').Not()
    return img.updateMask(clear)


def build_cloudfree_s2sr_col(aoi, start_date, end_date):
    """Return the collection with cloud and cloud shadow masks applied."""
    col = (get_s2_sr_cld_col(aoi, start_date, end_date)
           .map(add_cld_shdw_mask)
           .map(apply_cld_shdw_mask))
    return col


#### Define water masking functions

In [7]:
def build_non_water_mask(aoi: ee.Geometry,
                         start_date: str,
                         end_date: str,
                         seasonality_months: int = SEASONALITY_MONTHS,
                         dw_prob_thresh: float = DW_PROB_THRESHOLD) -> ee.Image:
    """Returns a boolean mask (1 = keep) for non-water using JRC + Dynamic World."""
    # Permanent water (JRC seasonality 0–12)
    jrc = (ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
           .select('seasonality')
           .unmask(0))
    perm_water = jrc.gte(seasonality_months)

    # Likely current water (Dynamic World water probability)
    dw = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
          .filterBounds(aoi)
          .filterDate(start_date, end_date)
          .select('water'))
    dw_water = dw.mean().gte(dw_prob_thresh)

    # Union water layers; keep non-water
    water_any = perm_water.max(dw_water)
    non_water = water_any.eq(0)
    return non_water

_NON_WATER = build_non_water_mask(AOI, START_DATE, END_DATE)

def apply_water_mask(img: ee.Image) -> ee.Image:
    """Apply water mask to ALL bands"""
    return img.updateMask(_NON_WATER)

### Process image collection and apply Cloud Masking

In [8]:
# Join S2SR Harmonized with S2 Cloud Probability
s2sr_cld_col = get_s2_sr_cld_col(AOI, START_DATE, END_DATE)

# Apply Cloud Masking
s2sr_cloudfree_col = build_cloudfree_s2sr_col(AOI, START_DATE, END_DATE)

# Calculate NDVI band
s2sr_cloudfree_col = s2sr_cloudfree_col.map(add_ndvi)

s2sr_clear_composite = s2sr_cloudfree_col.median().clip(LLBN)

ndvi = ee.Image(s2sr_clear_composite.select("NDVI"))
ndvi_water_mask = apply_water_mask(ndvi)


#### Visualize composite images

In [9]:
ndvi_vis = {
    'min': -0.2,
    'max': 0.8,
    'palette': [
        '#a50026', '#d73027', '#f46d43', '#fdae61',
        '#fee08b', '#d9ef8b', '#a6d96a', '#66bd63',
        '#1a9850', '#006837'
    ]
}

rgb_vis = {
    "bands": ["B4", "B3", "B2"],  # NIR, Red, Green
    "min": 0,
    "max": 3000,
}

fcc_vis = {
    "bands": ["B8", "B4", "B3"],  # NIR, Red, Green
    "min": 0,
    "max": 3000,
    "gamma": 1.4
}

swir_fcc_vis = {
    "bands": ["B12", "B8A", "B4"],  # SWIR2, NIR, Red
    "min": 0,
    "max": 3000,
    "gamma": 1.4
}


Map.addLayer(s2sr_clear_composite, fcc_vis, 'S2_SR FCC Composite', False)
Map.addLayer(s2sr_clear_composite, swir_fcc_vis, 'S2_SR SWIR FCC Composite', False)
Map.addLayer(s2sr_clear_composite, rgb_vis, 'S2_SR Cloudless Composite', True)
Map.addLayer(ndvi_water_mask, ndvi_vis, 'S2_SR NDVI Watermasked', True)

Map.centerObject(LLBN, 12)
Map

Map(center=[0.21352017398763973, 37.43775388457388], controls=(WidgetControl(options=['position', 'transparent…

#### Export Cloud-Optimized GeoTIFF to Google Drive

In [269]:
nodata = -9999
img_for_export = ndvi_water_mask.unmask(value=nodata, sameFootprint=False)

task = ee.batch.Export.image.toDrive(
    image=img_for_export,
    description='ndvi_water_masked_cog',
    folder='ee_exports',              
    fileNamePrefix='pvn_pillar4_ndvi_cog',
    region=LLBN,                      
    scale=10,                        
    crs='EPSG:4326',
    maxPixels=1e13,
    fileFormat='GeoTIFF',
    formatOptions={
        'cloudOptimized': True,
        'noData': nodata
    }
)
task.start()